*0.2 Math / ML basics*

# norms

**The situation.** Your vector database is configured for dot-product search because it is the fastest option. A second embedding model is added for a new language. Its vectors are not unit length, and every search that includes them is now wrong — but nothing errors, so it takes weeks to notice.

**The norm.** The *norm* of a vector is its length: the square root of the sum of the squares of its numbers. A vector with norm 1 is *unit length* or *normalised*. Dividing a vector by its own norm makes it unit length without changing its direction. Do that once, at index time, and dot product becomes cosine.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Measure, then normalise.** OpenAI's embeddings come out at length 1. TF-IDF vectors do not. `sklearn.preprocessing.normalize` fixes them.

In [2]:
import numpy as np
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

client = OpenAI(timeout=30)
texts = [
    "Reset your password from the login page.",
    "Password rules. Login rules. Refund policy. " * 50,
]

openai_vectors = []
for item in client.embeddings.create(model="text-embedding-3-small", input=texts).data:
    openai_vectors.append(item.embedding)
openai_vectors = np.array(openai_vectors, dtype=np.float32)
tfidf_vectors = TfidfVectorizer(norm=None).fit_transform(texts).toarray()

print(
    "OpenAI norms:  ", np.round(np.linalg.norm(openai_vectors, axis=1), 3), "← already unit length"
)
print("TF-IDF norms:  ", np.round(np.linalg.norm(tfidf_vectors, axis=1), 3), "← lengths differ")
unit_tfidf = normalize(tfidf_vectors)
print("after normalize:", np.round(np.linalg.norm(unit_tfidf, axis=1), 3))
assert np.allclose(np.linalg.norm(unit_tfidf, axis=1), 1.0)

OpenAI norms:   [1. 1.] ← already unit length
TF-IDF norms:   [  3.446 186.091] ← lengths differ
after normalize: [1. 1.]


**Reading the output.** Both OpenAI vectors have norm 1.0. The TF-IDF vectors have very different norms, and the long one is much longer. After `normalize` both are 1.0 — safe to search with dot product.

**Proof that it is the same as cosine.** Dot product on the normalised vectors equals cosine on the originals.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

dot_after = float(unit_tfidf[0] @ unit_tfidf[1])
cosine_before = float(cosine_similarity(tfidf_vectors[:1], tfidf_vectors[1:])[0, 0])
print("dot product of normalised vectors:", round(dot_after, 4))
print("cosine of the original vectors:   ", round(cosine_before, 4))
assert abs(dot_after - cosine_before) < 1e-6

dot product of normalised vectors: 0.1559
cosine of the original vectors:    0.1559


**The rule to remember.** Normalise at index time, search with dot product. Check the norm of a new model's output before it goes near an index.

| Use it when | Don't when | Instead use |
|---|---|---|
| any vector going into a dot-product index; mixing sources | the model documents unit-length output and you have checked it | nothing — already done |

**Watch out**
- A zero vector cannot be normalised (division by zero); `normalize` returns zeros silently, and the row will never match anything.
- Norms also flag bad data: an embedding with norm 0 or 40 when the rest are 1 is a bug upstream.
- L2 (this item) is the norm for similarity; L1 (sum of absolute values) appears in regularisation later.